In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')

In [3]:
#movies.head()
#credits.head()

In [4]:
#merge(join 2 dataset based on columns)
ds=pd.merge(movies,credits,left_on='id',right_on='movie_id',how='inner')

In [5]:
ds.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')

In [6]:
ds=ds[['id','title_x','genres','keywords','overview','production_companies','tagline','cast','crew']]

In [7]:
ds.rename(columns={'title_x':'Title'},inplace=True)

In [8]:
#Function which receives string of list of dict 
#converts to list and iterate all dict and returns comma seperated list of 'name' from dict
import ast #abstract syntax string
def get_names(mystring):
    geners=ast.literal_eval(mystring)
    words=[]
    for g in geners:
        words.append(g['name'])
    words=','.join(words)
    return words

In [9]:
ds['genres']=ds['genres'].apply(get_names)

In [10]:
ds['keywords']=ds['keywords'].apply(get_names)

In [11]:
ds['production_companies']=ds['production_companies'].apply(get_names)

In [12]:
ds['cast']=ds['cast'].apply(get_names)

In [13]:
ds['crew']=ds['crew'].apply(get_names)

In [14]:
ds.iloc[0]['overview']

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [15]:
def remove_spaces_with_comma(sentence):
    sentence=str(sentence)
    sentence=sentence.replace(',','')
    sentence=sentence.replace(' ','')
    return sentence

In [16]:
ds['overview']=ds['overview'].apply(remove_spaces_with_comma)

In [17]:
ds['tagline']=ds['tagline'].apply(remove_spaces_with_comma)

In [18]:
ds['tags']=ds['genres']+','+ds['keywords']+','+ds['overview']+','+ds['production_companies']+','+ds['tagline']+','+ds['cast']+','+ds['crew']

In [19]:
ds['tags']=ds['tags'].apply(lambda val: val.lower())

In [20]:
ds=ds[['id','Title','tags']]

In [21]:
def get_unique_words(original_string):
    # Step 1: Split, strip, and remove duplicates using set
    words = [word.strip() for word in original_string.split(',')]
    unique_words = list(set(words))
    # Optional: sort if you want consistent output
    unique_words.sort()
    # Step 2: Join back into a comma-separated string
    new_string = ','.join(unique_words)
    return new_string

In [22]:
ds['tags']=ds['tags'].apply(get_unique_words)

In [23]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [24]:
def remove_stop_words(text):
    # Get the English stopwords list
    stop_words = set(stopwords.words('english'))
    # Tokenize the sentence
    words = text.split(',')
    # Remove stopwords
    filtered_words = [word for word in words if word.lower() not in stop_words]
    # # Convert back to string
    filtered_text = ",".join(filtered_words)
    return filtered_text

In [25]:
ds['tags']=ds['tags'].apply(remove_stop_words)

In [26]:
ds['tags']=ds['tags'].apply(lambda val: val.replace(' ','_'))

In [27]:
ds.shape

(4803, 3)

In [28]:
ds.to_csv("D:\Github\python-data-preprocessing\Movie_Dataset/Cleaned_dataset.csv")